# CIFAR-10 Single Instance Overfitting Verification

This notebook verifies that the model successfully overfits a single CIFAR-10 image.

**Expected behavior:**
- Generated images should closely match the target image
- MSE between generated and target should be very low (<0.01)
- Visual comparison should show near-identical images

## Setup

### Training command:
```bash
python train_cifar10_single.py --max_steps 5000 --image_index 0
```

In [ ]:
# Navigate to PixNerd folder
import os
import sys

NOTEBOOK_DIR = os.getcwd()
print(f"Starting directory: {NOTEBOOK_DIR}")

# Navigate to PixNerd folder (where src/ lives)
PIXNERD_DIR = os.path.join(NOTEBOOK_DIR, "PixNerd")
if os.path.exists(PIXNERD_DIR):
    os.chdir(PIXNERD_DIR)
    print(f"Changed to: {os.getcwd()}")
elif os.path.basename(NOTEBOOK_DIR) == "PixNerd":
    print(f"Already in PixNerd directory: {NOTEBOOK_DIR}")
else:
    parent = os.path.dirname(NOTEBOOK_DIR)
    pixnerd_in_parent = os.path.join(parent, "PixNerd")
    if os.path.exists(pixnerd_in_parent):
        os.chdir(pixnerd_in_parent)
        print(f"Changed to: {os.getcwd()}")
    else:
        print(f"WARNING: Could not find PixNerd folder")

if os.path.exists("src"):
    print("Found src/ directory")
else:
    print("ERROR: src/ directory not found!")

In [ ]:
from pathlib import Path
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

# Paths
PIXNERD_ROOT = Path(os.getcwd())

# ============================================================
# CHECKPOINT PATH - UPDATE THIS TO YOUR TRAINED MODEL
# ============================================================
EXP_DIR = PIXNERD_ROOT / "workdirs" / "exp_cifar10_single_overfit"
CKPT_PATH = EXP_DIR / "checkpoints" / "last.ckpt"
TARGET_IMG_PATH = EXP_DIR / "target_image.png"
CLASS_LABEL_PATH = EXP_DIR / "training_class.txt"
# ============================================================

OUTPUT_DIR = PIXNERD_ROOT / "outputs" / "cifar10_single"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Model config (must match train_cifar10_single.py)
NUM_CLASSES = 10
BASE_RES = 32
PATCH_SIZE = 8
HIDDEN_SIZE = 512
DECODER_HIDDEN_SIZE = 64
NUM_ENCODER_BLOCKS = 8
NUM_DECODER_BLOCKS = 2
NUM_GROUPS = 8

CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

print(f"Checkpoint path: {CKPT_PATH}")
print(f"Checkpoint exists: {CKPT_PATH.exists()}")
print(f"Target image exists: {TARGET_IMG_PATH.exists()}")
print(f"Class label file exists: {CLASS_LABEL_PATH.exists()}")
print(f"Device: {DEVICE}")

## Load Target Image

In [ ]:
# Load target image
target_pil = Image.open(TARGET_IMG_PATH)
target_np = np.array(target_pil)
target_tensor = torch.from_numpy(target_np).permute(2, 0, 1).float() / 255.0
target_tensor = (target_tensor - 0.5) / 0.5  # Normalize to [-1, 1]

print(f"Target image shape: {target_np.shape}")
print(f"Target tensor shape: {target_tensor.shape}")

plt.figure(figsize=(4, 4))
plt.imshow(target_np)
plt.title("Target Image (to be memorized)")
plt.axis('off')
plt.show()

## Build Model

In [ ]:
# Import PixNerd components
from src.models.autoencoder.pixel import PixelAE
from src.models.conditioner.class_label import LabelConditioner
from src.models.transformer.pixnerd_c2i_heavydecoder import PixNerDiT
from src.diffusion.flow_matching.scheduling import LinearScheduler
from src.diffusion.flow_matching.sampling import EulerSampler, ode_step_fn
from src.diffusion.base.guidance import simple_guidance_fn
from src.diffusion.flow_matching.training import FlowMatchingTrainer
from src.callbacks.simple_ema import SimpleEMA
from src.lightning_model import LightningModel
from src.models.autoencoder.base import fp2uint8

print("Imports successful!")

In [ ]:
print("Initializing model components...")

main_scheduler = LinearScheduler()

vae = PixelAE(scale=1.0)

conditioner = LabelConditioner(num_classes=NUM_CLASSES)

denoiser = PixNerDiT(
    in_channels=3,
    patch_size=PATCH_SIZE,
    num_groups=NUM_GROUPS,
    hidden_size=HIDDEN_SIZE,
    decoder_hidden_size=DECODER_HIDDEN_SIZE,
    num_encoder_blocks=NUM_ENCODER_BLOCKS,
    num_decoder_blocks=NUM_DECODER_BLOCKS,
    num_classes=NUM_CLASSES,
)

# Sampler with low guidance for overfitting
sampler = EulerSampler(
    num_steps=50,
    guidance=1.0,  # No CFG for overfitting
    guidance_interval_min=0.0,
    guidance_interval_max=1.0,
    scheduler=main_scheduler,
    w_scheduler=LinearScheduler(),
    guidance_fn=simple_guidance_fn,
    step_fn=ode_step_fn,
)

trainer_stub = FlowMatchingTrainer(
    scheduler=main_scheduler,
    lognorm_t=True,
    timeshift=1.0,
)

ema_tracker = SimpleEMA(decay=0.9999)

model = LightningModel(
    vae=vae,
    conditioner=conditioner,
    denoiser=denoiser,
    diffusion_trainer=trainer_stub,
    diffusion_sampler=sampler,
    ema_tracker=ema_tracker,
    optimizer=None,
    lr_scheduler=None,
    eval_original_model=False,
)

model.eval()
model.to(DEVICE)
print(f"Model initialized and moved to {DEVICE}")

## Load Checkpoint

In [ ]:
print(f"Loading checkpoint from: {CKPT_PATH}")
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)
print(f"Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")
print("Checkpoint loaded successfully!")

## Helper Functions

In [ ]:
@torch.no_grad()
def sample_single(
    class_label: int,
    num_samples: int = 1,
    seed: int = 42,
    num_steps: int = 50,
    guidance: float = 1.0,
):
    """Generate samples for a specific class."""
    torch.manual_seed(seed)
    
    # Configure sampler
    model.diffusion_sampler.guidance = guidance
    model.diffusion_sampler.num_steps = num_steps
    
    # Generate noise
    noise = torch.randn(num_samples, 3, 32, 32, device=DEVICE)
    
    # Get condition
    labels = [class_label] * num_samples
    condition, uncondition = model.conditioner(labels)
    condition = condition.to(DEVICE)
    uncondition = uncondition.to(DEVICE)
    
    # Sample
    samples = model.diffusion_sampler(
        model.ema_denoiser,
        noise,
        condition,
        uncondition,
    )
    
    # Decode
    images = model.vae.decode(samples)
    images = torch.clamp(images, -1.0, 1.0)
    images_uint8 = fp2uint8(images)
    
    return images_uint8.cpu(), samples.cpu()


def compute_metrics(generated, target):
    """Compute MSE and PSNR between generated and target images."""
    # Convert to [0, 1] range
    if generated.max() > 1:
        generated = generated.float() / 255.0
    if target.max() > 1:
        target = target.float() / 255.0
    
    mse = F.mse_loss(generated, target).item()
    psnr = 10 * np.log10(1.0 / (mse + 1e-10))
    
    return mse, psnr


print("Helper functions defined.")

## Find Correct Class Label

If the class label file doesn't exist, we'll test all 10 classes and find the one with lowest MSE.

In [ ]:
# Try to load training class from file
if CLASS_LABEL_PATH.exists():
    with open(CLASS_LABEL_PATH) as f:
        CLASS_LABEL = int(f.read().strip())
    print(f"Loaded class label from file: {CLASS_LABEL} ({CIFAR10_CLASSES[CLASS_LABEL]})")
else:
    print("Class label file not found. Testing all 10 classes to find the best match...")
    print()
    
    tgt_tensor = torch.from_numpy(target_np).permute(2, 0, 1).float() / 255.0
    
    class_mses = []
    for class_idx in range(10):
        samples_uint8, _ = sample_single(
            class_label=class_idx,
            num_samples=1,
            seed=42,
            num_steps=50,
            guidance=1.0,
        )
        gen_tensor = samples_uint8[0].float() / 255.0
        mse, _ = compute_metrics(gen_tensor, tgt_tensor)
        class_mses.append(mse)
        print(f"  Class {class_idx} ({CIFAR10_CLASSES[class_idx]:>10}): MSE = {mse:.6f}")
    
    CLASS_LABEL = np.argmin(class_mses)
    print()
    print(f"Best class: {CLASS_LABEL} ({CIFAR10_CLASSES[CLASS_LABEL]}) with MSE = {class_mses[CLASS_LABEL]:.6f}")

print(f"\nUsing class label: {CLASS_LABEL} ({CIFAR10_CLASSES[CLASS_LABEL]})")

## Generate and Compare

Generate samples with different seeds and compare to target.

In [ ]:
# Generate multiple samples
num_samples = 5
all_samples = []
all_mses = []
all_psnrs = []

tgt_tensor = torch.from_numpy(target_np).permute(2, 0, 1).float() / 255.0

for seed in range(num_samples):
    samples_uint8, samples_raw = sample_single(
        class_label=CLASS_LABEL,
        num_samples=1,
        seed=seed,
        num_steps=100,  # More steps for better quality
        guidance=1.0,
    )
    all_samples.append(samples_uint8[0])
    
    # Compute metrics
    gen_tensor = samples_uint8[0].float() / 255.0
    mse, psnr = compute_metrics(gen_tensor, tgt_tensor)
    all_mses.append(mse)
    all_psnrs.append(psnr)
    print(f"Seed {seed}: MSE={mse:.6f}, PSNR={psnr:.2f} dB")

print(f"\nAverage MSE: {np.mean(all_mses):.6f}")
print(f"Average PSNR: {np.mean(all_psnrs):.2f} dB")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, num_samples + 1, figsize=(3 * (num_samples + 1), 6))

# Top row: target and generated samples
axes[0, 0].imshow(target_np)
axes[0, 0].set_title("Target")
axes[0, 0].axis('off')

for i, sample in enumerate(all_samples):
    sample_np = sample.permute(1, 2, 0).numpy()
    axes[0, i + 1].imshow(sample_np)
    axes[0, i + 1].set_title(f"Seed {i}\nMSE={all_mses[i]:.4f}")
    axes[0, i + 1].axis('off')

# Bottom row: difference maps
axes[1, 0].axis('off')
axes[1, 0].set_title("Difference Maps")

for i, sample in enumerate(all_samples):
    sample_np = sample.permute(1, 2, 0).numpy().astype(float)
    diff = np.abs(sample_np - target_np.astype(float))
    diff_normalized = (diff / 255.0 * 5).clip(0, 1)  # Amplify for visibility
    axes[1, i + 1].imshow(diff_normalized)
    axes[1, i + 1].set_title(f"Diff (5x amp)")
    axes[1, i + 1].axis('off')

plt.suptitle(f"Single Instance Overfitting - Class: {CIFAR10_CLASSES[CLASS_LABEL]}", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overfitting_comparison.png", dpi=150)
plt.show()

## Metrics Summary

In [ ]:
print("="*60)
print("OVERFITTING VERIFICATION SUMMARY")
print("="*60)
print(f"Class: {CLASS_LABEL} ({CIFAR10_CLASSES[CLASS_LABEL]})")
print(f"Number of samples tested: {num_samples}")
print(f"Average MSE: {np.mean(all_mses):.6f}")
print(f"Average PSNR: {np.mean(all_psnrs):.2f} dB")
print(f"Best MSE: {np.min(all_mses):.6f} (seed {np.argmin(all_mses)})")
print(f"Best PSNR: {np.max(all_psnrs):.2f} dB (seed {np.argmax(all_psnrs)})")
print()

# Quality assessment
avg_mse = np.mean(all_mses)
if avg_mse < 0.001:
    print("EXCELLENT: Model has perfectly memorized the image!")
elif avg_mse < 0.01:
    print("GOOD: Model has mostly memorized the image.")
elif avg_mse < 0.05:
    print("FAIR: Model is learning but needs more training.")
else:
    print("POOR: Model has not yet memorized the image. Train longer!")

print("="*60)

## Super-Resolution Test (Optional)

Test if the overfitted model can also generate at higher resolution.

In [ ]:
def set_decoder_scale(scale: float):
    """Set NF decoder patch scaling for super-resolution."""
    for net in [model.denoiser, getattr(model, "ema_denoiser", None)]:
        if net is None:
            continue
        net.decoder_patch_scaling_h = scale
        net.decoder_patch_scaling_w = scale


@torch.no_grad()
def sample_superres(
    class_label: int,
    height: int = 128,
    width: int = 128,
    seed: int = 42,
    num_steps: int = 50,
):
    """Generate super-resolution sample."""
    torch.manual_seed(seed)
    
    scale = height / 32.0
    set_decoder_scale(scale)
    
    model.diffusion_sampler.guidance = 1.0
    model.diffusion_sampler.num_steps = num_steps
    
    noise = torch.randn(1, 3, height, width, device=DEVICE)
    
    condition, uncondition = model.conditioner([class_label])
    condition = condition.to(DEVICE)
    uncondition = uncondition.to(DEVICE)
    
    samples = model.diffusion_sampler(
        model.ema_denoiser,
        noise,
        condition,
        uncondition,
    )
    
    images = model.vae.decode(samples)
    images = torch.clamp(images, -1.0, 1.0)
    images_uint8 = fp2uint8(images)
    
    # Reset scale
    set_decoder_scale(1.0)
    
    return images_uint8.cpu()


# Generate at different resolutions
print("Generating super-resolution samples...")

best_idx = np.argmin(all_mses)
img_32 = all_samples[best_idx]  # Best 32x32
img_64 = sample_superres(CLASS_LABEL, 64, 64, seed=best_idx, num_steps=100)
img_128 = sample_superres(CLASS_LABEL, 128, 128, seed=best_idx, num_steps=100)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(target_np)
axes[0].set_title("Target 32x32")
axes[0].axis('off')

axes[1].imshow(img_32.permute(1, 2, 0).numpy())
axes[1].set_title("Generated 32x32")
axes[1].axis('off')

axes[2].imshow(img_64[0].permute(1, 2, 0).numpy())
axes[2].set_title("Super-Res 64x64")
axes[2].axis('off')

axes[3].imshow(img_128[0].permute(1, 2, 0).numpy())
axes[3].set_title("Super-Res 128x128")
axes[3].axis('off')

plt.suptitle("Single Instance at Multiple Resolutions", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overfitting_superres.png", dpi=150)
plt.show()

In [ ]:
print("Done!")
print(f"Outputs saved to: {OUTPUT_DIR}")